# SQL Worksheet — Week3

Use the following tables from the Riva Data Platform:

- `rivadataplatform.dataproduct.dim_batch`
- `rivadataplatform.dataproduct.dim_class`
- `rivadataplatform.dataproduct.fact_attendance`
- `rivadataplatform.dataproduct.dim_student`
- `rivadataplatform.dataproduct.dim_date`

**Instructions**
- Write SQL for each question.
- Do not modify the source data.
- Use clear aliases where JOINs are involved.
- Unless a question specifically asks for a particular column, select only the columns needed to answer it.


## Tables / Relationships

Useful keys:
- `dim_student.student_key` ↔ `fact_attendance.student_key`
- `dim_class.class_key` ↔ `fact_attendance.class_key`
- `dim_batch.batch_key` ↔ `fact_attendance.batch_key`
- `dim_class.batch_id` ↔ `dim_batch.batch_id`

## Question 1 — Filtered Student Risk Profile
Join students to attendance and class data, filter to students whose names start with `A` and whose attendance is Late or Absent, then group by student and topic. Return issue count, class date range, and a null-safe topic label.

In [0]:
--Write your code here
 select 
   s.student_key,
   s.student_name,

   coalesce(nullif(c.topic, 'nan'), 'topic not assigned') as topic,

   count(*) as issue_count,
   min(c.class_date) as first_class_date,
   max(c.class_date) as last_class_date
   
   from rivadataplatform.dataproduct.fact_attendance as a
   join rivadataplatform.dataproduct.dim_students as s
     on a.student_key = s.student_key
   join rivadataplatform.dataproduct.dim_class as c
     on a.class_key = c.class_key
  where s.student_name like 'A%'
  group by s.student_key,s.student_name,
  coalesce(nullif(c.topic, 'nan'), 'topic not assigned')
  order by
  s.student_name,
  issue_count DESC;

## Question 2 — Date-Range Attendance Detail
Join attendance to students, classes, batches, and `dim_date`. Return records whose class date falls between the batch start and end dates, showing student, batch, topic, calendar day, and status. Exclude records with a null status and sort chronologically.

In [0]:
--Write your code here

select 
  
  s.student_name,
  b.batch_name,
  c.topic,
  d.date,
  a.attendance_status

  from rivadataplatform.dataproduct.fact_attendance as a

  join rivadataplatform.dataproduct.dim_students as s
    on a.student_key = s.student_key

  join rivadataplatform.dataproduct.dim_batch as b
    on a.batch_key = b.batch_key

  join rivadataplatform.dataproduct.dim_class as c
      on a.class_key = c.class_key

   join rivadataplatform.dataproduct.dim_date as d
      on a.date_key = d.date_key

   where c.class_date >= b.start_date
   and c.class_date <= b.end_date
   and a.attendance_status is not null      

   order by c.class_date;

  

## Question 3 — Conditional Batch Scorecard
Join attendance to batch and date dimensions and return one row per batch and calendar month. Calculate Present, Late, Absent, total records, and distinct students. Use conditional aggregation and keep only months containing at least one Absent record.

In [0]:
--Write your code here

select 
 b.batch_name,
 d.year,
 d.month,
 d.month_name,

 sum(case when a.attendance_status = 'present' then 1
     else 0 
     end) as present_count,

  sum(case when a.attendance_status = 'late' then 1
  else 0
  end) as late_count,
  sum(case when a.attendance_status = 'absent' then 1
  else 0
  end) as absent_count,

 count(*) as total_records,
 count(distinct a.student_key) as distinct_students
 from rivadataplatform.dataproduct.fact_attendance as a

 join rivadataplatform.dataproduct.dim_batch as b
   on a.batch_key = b.batch_key

 join rivadataplatform.dataproduct.dim_date as d
   on a.date_key = d.date_key

 group by
    b.batch_name,
    d.year,
    d.month,
    d.month_name

 having sum(
    case when a.attendance_status = 'absent' then 1
    else 0 
    end
 ) >=1

 order by 
 d.year,
 d.month;

## Question 4 — Class Coverage Including Empty Classes
Use `dim_class` as the driving table and left join attendance, students, batch, and date dimensions. Group by class and return class metadata, distinct students, total records, and average `attendance_count`, showing zero/null-safe values for classes without attendance.

In [0]:
--Write your code here
select 
 c.class_id,
 c.class_date,
 c.class_day,
 c.topic,
 c.instructor,
 c.status,
 b.batch_name,

 count(distinct s.student_key) as distinct_students,
 
 count(a.attendance_id) as total_records,

 coalesce(
    avg(a.attendance_count), 0
 ) as avg_attendance_count

 from rivadataplatform.dataproduct.dim_class c

 left join rivadataplatform.dataproduct.fact_attendance a
 on c.class_key = a.class_key

 left join rivadataplatform.dataproduct.dim_students s
 on a.student_key = s.student_key

 left join rivadataplatform.dataproduct.dim_batch b
 on c.batch_id = b.batch_id

 left join rivadataplatform.dataproduct.dim_date d
 on c.class_date = d.date

 group by
 c.class_id,
 c.class_date,
 c.class_day,
 c.topic,
 c.instructor,
 c.status,
 b.batch_name

 order by c.class_date;



## Question 5 — Average Attendance by Topic and Status
Join attendance to class and batch dimensions. Group by batch, null-safe topic, and attendance status, and calculate average `attendance_count`, total records, and distinct students. Exclude null/blank statuses and sort by average descending.

In [0]:
--Write your code here

select 
 b.batch_name,
 a.attendance_status,

 coalesce(c.topic,'topic not assigned') as topic,
 avg(a.attendance_count) as avg_attendance_count,

 count(*) as total_records,

 count(distinct a.student_key) as distinct_students

 from rivadataplatform.dataproduct.fact_attendance as a

 join rivadataplatform.dataproduct.dim_class c
    on a.class_key = c.class_key

 join rivadataplatform.dataproduct.dim_batch b
     on a.batch_key = b.batch_key

 where a.attendance_status is not null
 and trim(a.attendance_status) <> ''
 group by 
 b.batch_name,
 a.attendance_status,
 c.topic
 order by avg_attendance_count desc;   

 

 

## Question 6 — High-Volume Students With Profile Gaps
Use a `LEFT JOIN` from students through attendance, classes, and batches. Group by student and batch, then return students with at least two records, including Present count, issue count, and a null-safe phone label. Order by issue count and total records.

In [0]:
--Write your code here

select 
s.student_name,
b.batch_name,

COALESCE(s.phone_no, 'Phone not available') AS phone,

    SUM(
        CASE
            WHEN a.attendance_status = 'Present' THEN 1
            ELSE 0
        END
    ) AS present_count,

    SUM(
        CASE
            WHEN a.attendance_status IN ('Late', 'Absent') THEN 1
            ELSE 0
        END
    ) AS issue_count,

count(a.attendance_id) as total_records

from rivadataplatform.dataproduct.dim_students s
left join rivadataplatform.dataproduct.fact_attendance a
on s.student_key = a.student_key
left join rivadataplatform.dataproduct.dim_class c
on a.class_key = c.class_key
left join rivadataplatform.dataproduct.dim_batch b
on a.batch_key = b.batch_key


group by 
s.student_name,
b.batch_name,
s.phone_no


having count (a.attendance_id)>=2
order by
issue_count desc,
total_records desc;
